## Normalizng Flows

#### The core idea behind normalising flows is deceptively simple: if we can express a complex distribution as an invertible transformation of a tractable one, then both Sampling and density evaluation reduce to applications of the change-of-variables formula. From  a base distribution $u\sim\mathcal{N}(o,\mathbf{I})$, and appying a diffemorphism $T$, the densoity of $\theta = T(\mathbf{u})$ is 
$$
  p_{\theta}(\theta) = p_{\mathbf{u}}\!\left(T^{-1}(\theta)\right) 
    \left|\det J_{T^{-1}}(\theta)\right|,
$$
#### A single transformation rarely suffices, but diffeomorphisms compose:  stacking $K$: layers as $T = T_K \circ \cdots \circ T_1$ yields
$$
   \log p_{\theta}(\theta) = \log p_{\mathbf{u}}(\mathbf{u}) 
    - \sum_{k=1}^{K} \log\left|\det J_{T_k}(\mathbf{z}_{k-1})\right|,
$$
#### where each added layer increases the family of representable distributions. For NPE, we need more than an unconditional density---we need $q_{\bm{\phi}}(\theta \mid \mathbf{y})$, a posterior that adapts to each observation. The solution is to condition the transformation itself on $y$:
$$
    q_{\bm{\phi}}(\theta \mid y) = p_{\mathbf{u}}\!\left(T^{-1}(\theta; y)\right) 
    \left|\det J_{T^{-1}}(\theta; y)\right|.
$$
#### Now different observations induce different posteriors through the same network weights. The computational obstacle in the above equation is the Jacobian determinant: a naïve implementation costs $\mathcal{O}(d^3)$ per layer, which becomes prohibitive even for modest parameter dimensions. The solution adopted by all modern flow architectures is to constrain the Jacobian to be triangular, so  that its determinant reduces to the product of diagonal entries, $\mathcal{O}(d)$ operations regardless of network depth.

#### **RealNVP** stands for Real-valued Non-Volume Preserving transformations. It is one of the foundational architectures for Normalising Flows. "2D" simply means we are operating on two-dimensional data.

In [1]:
import torch 
import torch.nn as nn
import torch.distributions as dist

In [2]:
class AffineCouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask):
        super().__init__()
        # The mask tells us which dimensions to keep unchanged (True) 
        # and which to transform (False)
        self.mask = mask 
        
        # This network predicts how to scale (s) and translate (t) the data.
        # Notice it outputs input_dim * 2 because we need both s and t!
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim * 2) 
        )

In [4]:
def forward(self, x):
        """ Data Space -> Latent Space (Used for Density Evaluation) """
        # 1. Apply mask to get the unchanged part of the data
        x_masked = x * self.mask
        
        # 2. Predict scale (s) and translation (t)
        s_and_t = self.net(x_masked)
        s, t = s_and_t.chunk(2, dim=1)
        
        # 3. Ensure we only apply s and t to the dimensions we are transforming
        s = s * (~self.mask)
        t = t * (~self.mask)
        
        # 4. The Inverse Transformation: T^{-1}(x) -> u
        u = (x - t) * torch.exp(-s)
        
        # 5. The Log-Determinant of the Jacobian
        log_det_J = -s.sum(dim=1) 
        
        return u, log_det_J

In [5]:
def inverse(self, u):
        """ Latent Space -> Data Space (Used for Sampling) """
        u_masked = u * self.mask
        s_and_t = self.net(u_masked)
        s, t = s_and_t.chunk(2, dim=1)
        
        s = s * (~self.mask)
        t = t * (~self.mask)
        
        # The Forward Transformation: T(u) -> x
        x = u * torch.exp(s) + t
        return x